# EEG · 03 · Low-level PCA + multitask (Experiment 3)
**Question:** does the low-level branch add information without hurting CLIP retrieval?

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config, vae_pca_model_path, load_json
from src.data import build_datamodule
cfg = load_config('configs/EEG/exp03_lowlevel_multitask.yaml')
dm = build_datamodule(cfg).prepare()

## PCA explained variance of the VAE latents
(Run `scripts/04_precompute_vae_pca.py --config configs/EEG/exp03_lowlevel_multitask.yaml` first.)

In [ ]:
subj = dm.subjects[0]
evr = load_json(str(vae_pca_model_path(cfg, subj)).replace('.pkl','.evr.json'))
plt.figure(figsize=(7,4))
plt.plot(np.cumsum(evr['explained_variance_ratio']))
plt.xlabel('PCA component'); plt.ylabel('cumulative explained variance')
plt.title(f'VAE-PCA explained variance ({subj})'); plt.grid(True); plt.show()

## Train the multitask model
Same `run_training` as `scripts/05_train_multitask.py` (resumable).

In [ ]:
from src.training import train_multitask
cfg['training']['epochs'] = 100
result = train_multitask(cfg, resume=None)
result['metrics']

## CLIP-only vs multitask retrieval

In [ ]:
import pandas as pd
clip_only = load_json('outputs/exp01_eeg_to_clip/metrics/test_metrics.json')['retrieval']
multitask = result['metrics']['test']['retrieval']
pd.DataFrame({'clip_only': clip_only, 'multitask': multitask})

## Low-level prediction quality (per-PCA-component Pearson)

In [ ]:
%matplotlib inline

In [ ]:
import torch
from src.evaluation import load_subject_matrices, per_component_pearson
from src.models import build_model_from_checkpoint
from src.utils import get_device
device = get_device(cfg.get('runtime.device','auto'))
model,_ = build_model_from_checkpoint(cfg, 'outputs/exp03_eeg_lowlevel_multitask/checkpoints/best.pt', device, dm.voxel_counts)
m = load_subject_matrices(cfg, dm, subj, 'test', want=('fmri','low'))
with torch.no_grad():
    x = torch.from_numpy(m.fmri).float().to(device)
    low_pred = model(x, subject=subj)['low'].cpu().numpy()
r = per_component_pearson(low_pred, m.low)
plt.figure(figsize=(7,4)); plt.plot(r); plt.axhline(0, color='k', lw=0.5)
plt.xlabel('PCA component'); plt.ylabel('Pearson r'); plt.title('Low-level prediction'); plt.show()
print('mean Pearson r = %.3f' % np.nanmean(r))

**Takeaway:** the multitask model should keep (or improve) CLIP retrieval while predicting low-level components above a mean/zero baseline (positive Pearson r, R²>0).